# step_combined

> Phase 2 combined step renderer: dual-column layout for Segment & Align

In [ ]:
#| default_exp components.step_combined

In [ ]:
#| export
from typing import Any

from fasthtml.common import Div, H2, H3, P, Span, Input

# DaisyUI components
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_styles, badge_sizes
from cjm_fasthtml_daisyui.components.feedback.alert import alert, alert_colors
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui, border_dui, ring_dui
from cjm_fasthtml_daisyui.utilities.border_radius import border_radius

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, h, min_h
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight, uppercase, tracking
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.borders import border
from cjm_fasthtml_tailwind.utilities.effects import opacity, ring
from cjm_fasthtml_tailwind.utilities.transitions_and_animation import transition, duration
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, justify, items, gap, grow
)
from cjm_fasthtml_tailwind.core.base import combine_classes

# Interaction context
from cjm_fasthtml_interactions.core.context import InteractionContext

# Local imports
from cjm_fasthtml_workflow_transcript_decomp.core.html_ids import StructureDecompHtmlIds
from cjm_fasthtml_workflow_transcript_decomp.routes.models import DecompUrls

## Column Headers with Mini-Stats

Each column has a header showing its title and a badge area for compact stats
that remain visible regardless of which column is active.

In [ ]:
#| export
def _render_column_header(
    title:str,  # Column title (e.g., "Text Decomposition")
    stats_id:str,  # HTML ID for the mini-stats badge area
    header_id:str,  # HTML ID for the column header container
) -> Any:  # Column header component
    """Render a column header with title and mini-stats badge."""
    return Div(
        Span(
            title,
            cls=combine_classes(
                font_size.sm, font_weight.bold,
                uppercase, tracking.wide,
                text_dui.base_content.opacity(50)
            )
        ),
        Span(
            "--",
            id=stats_id,
            cls=combine_classes(badge, badge_styles.ghost, badge_sizes.sm)
        ),
        id=header_id,
        cls=combine_classes(
            flex_display, justify.between, items.center,
            p(3), bg_dui.base_200,
            border_dui.base_300, border.b()
        )
    )

## Shared Chrome Containers

Stable-ID containers for keyboard hints, toolbar, controls, and footer.
These receive OOB innerHTML swaps when focus switches between columns.
Initially rendered with placeholder content.

In [ ]:
#| export
def _render_shared_chrome() -> tuple:  # (hints, toolbar, controls, footer)
    """Render shared chrome containers with placeholder content."""
    hints = Div(
        P(
            "Keyboard hints will appear here when a column is active.",
            cls=combine_classes(font_size.sm, text_dui.base_content.opacity(50))
        ),
        id=StructureDecompHtmlIds.SHARED_HINTS,
        cls=str(p(2))
    )

    toolbar = Div(
        P(
            "Toolbar actions will appear here based on the active column.",
            cls=combine_classes(font_size.sm, text_dui.base_content.opacity(50))
        ),
        id=StructureDecompHtmlIds.SHARED_TOOLBAR,
        cls=str(p(2))
    )

    controls = Div(
        P(
            "Column-specific controls will appear here.",
            cls=combine_classes(font_size.sm, text_dui.base_content.opacity(50))
        ),
        id=StructureDecompHtmlIds.SHARED_CONTROLS,
        cls=str(p(2))
    )

    footer = Div(
        P(
            "Footer with progress and timestamp details will appear here.",
            cls=combine_classes(font_size.sm, text_dui.base_content.opacity(50))
        ),
        id=StructureDecompHtmlIds.SHARED_FOOTER,
        cls=combine_classes(
            p(4), bg_dui.base_100,
            border_dui.base_300, border.t(),
            flex_display, justify.between, items.center
        )
    )

    return hints, toolbar, controls, footer

## Column Containers

Left column (60%, text decomposition) and right column (40%, VAD alignment).
Each column has a stable HTML ID for HTMX content swapping, a header with
mini-stats, a scrollable content area, and visual indicators for active/inactive
state. Columns stack vertically below the `lg` breakpoint.

In [ ]:
#| export
def _render_decomp_column(
    is_active:bool=True,  # Whether this column is initially active
) -> Any:  # Left column component
    """Render the left decomposition column with placeholder content."""
    active_cls = combine_classes(ring(2), ring_dui.primary) if is_active else str(opacity(70))

    return Div(
        _render_column_header(
            title="Text Decomposition",
            stats_id=StructureDecompHtmlIds.DECOMP_MINI_STATS,
            header_id=StructureDecompHtmlIds.DECOMP_COLUMN_HEADER,
        ),
        # Scrollable content area (placeholder)
        Div(
            Div(
                Span("Placeholder", cls=str(font_weight.bold)),
                ": The card stack segment editor will render here.",
                cls=combine_classes(alert, alert_colors.info, m.b(4))
            ),
            P(
                "This column will contain the full text decomposition UI "
                "with card stack viewport, split/merge controls, and token selector.",
                cls=combine_classes(font_size.sm, text_dui.base_content.opacity(60))
            ),
            cls=combine_classes(grow(), overflow.y.auto, p(4))
        ),
        id=StructureDecompHtmlIds.DECOMP_COLUMN,
        cls=combine_classes(
            w.full, w('[60%]').lg,
            min_h(0),
            flex_display, flex_direction.col,
            bg_dui.base_100, border_dui.base_300, border(1),
            border_radius.box,
            overflow.hidden,
            active_cls,
            transition.all, duration._200,
        )
    )

In [ ]:
#| export
def _render_alignment_column(
    is_active:bool=False,  # Whether this column is initially active
) -> Any:  # Right column component
    """Render the right alignment column with placeholder content."""
    active_cls = combine_classes(ring(2), ring_dui.secondary) if is_active else str(opacity(70))

    return Div(
        _render_column_header(
            title="VAD Alignment",
            stats_id=StructureDecompHtmlIds.ALIGNMENT_MINI_STATS,
            header_id=StructureDecompHtmlIds.ALIGNMENT_COLUMN_HEADER,
        ),
        # Scrollable content area (placeholder)
        Div(
            Div(
                Span("Placeholder", cls=str(font_weight.bold)),
                ": The VAD chunk timeline and alignment UI will render here.",
                cls=combine_classes(alert, alert_colors.info, m.b(4))
            ),
            P(
                "This column will contain VAD chunks from audio analysis, "
                "with keyboard-driven linking to text segments.",
                cls=combine_classes(font_size.sm, text_dui.base_content.opacity(60))
            ),
            cls=combine_classes(grow(), overflow.y.auto, p(4))
        ),
        id=StructureDecompHtmlIds.ALIGNMENT_COLUMN,
        cls=combine_classes(
            w.full, w('[40%]').lg,
            min_h(0),
            flex_display, flex_direction.col,
            bg_dui.base_100, border_dui.base_300, border(1),
            border_radius.box,
            overflow.hidden,
            active_cls,
            transition.all, duration._200,
        )
    )

## Main Combined Step Renderer

Top-level step renderer for the merged Phase 2. Composes the shared chrome,
dual-column layout, and a hidden input tracking the active column.

In [ ]:
#| export
def render_combined_step(
    ctx:InteractionContext,  # Interaction context with state and data
    decomp_urls:DecompUrls=None,  # URL bundle for decomposition routes
) -> Any:  # FastHTML component with full dual-column layout
    """Render Phase 2: Combined Segment & Align step with dual-column layout."""
    decomp_urls = decomp_urls or DecompUrls()

    # Shared chrome containers (hints, toolbar, controls, footer)
    hints, toolbar, controls, footer = _render_shared_chrome()

    # Hidden input tracking active column (decomp or align)
    active_column_input = Input(
        type="hidden",
        id=StructureDecompHtmlIds.ACTIVE_COLUMN_INPUT,
        name="active_column",
        value="decomp",
    )

    return Div(
        # Header
        Div(
            H2("Segment & Align", cls=combine_classes(font_size._3xl, font_weight.bold)),
            P(
                "Decompose text into segments and align with audio timestamps.",
                cls=combine_classes(text_dui.base_content.opacity(70), m.b(4))
            ),
        ),

        # Shared keyboard hints
        hints,

        # Shared toolbar and controls
        toolbar,
        controls,

        # Dual-column content area
        Div(
            _render_decomp_column(is_active=True),
            _render_alignment_column(is_active=False),
            cls=combine_classes(
                grow(),
                min_h(0),
                flex_display,
                flex_direction.col,
                flex_direction.row.lg,
                gap(4),
                overflow.hidden,
                p(1),
            )
        ),

        # Shared footer
        footer,

        # Hidden active column state
        active_column_input,

        id=StructureDecompHtmlIds.DECOMP_CONTAINER,
        cls=combine_classes(
            w.full, h.full,
            flex_display, flex_direction.col,
            p(4)
        )
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()